# Exemplo: verificação de carregamentos pelo Método I

Este notebook cria um `PilarCircularPreenchido` e usa o `MetodoI` para verificar se uma lista de carregamentos passa ou não na interação axial + flexão.

As unidades usadas pelo pacote são `N` para força e `N.mm` para momento. Para deixar o exemplo mais amigável, os carregamentos são definidos em `kN` e `kN.m` e convertidos antes da verificação.

## 1. Importação das classes

Este exemplo assume que a biblioteca `MY_PACKAGE` já está instalada no ambiente Python usado pelo notebook.

In [1]:
from MY_PACKAGE.domain.value_objects.pilar_misto_circular import PilarCircularPreenchido
from MY_PACKAGE.domain.value_objects.ObjetoConcreto import ConcretoNormal
from MY_PACKAGE.domain.value_objects.ObjetoAco import AcoEstrutural, AcoArmadura
from MY_PACKAGE.domain.services.metodo_I import MetodoI

## 2. Criação do pilar circular misto

A seção usada aqui é a mesma ordem de grandeza do exemplo básico de pilar circular: tubo de `600 mm`, espessura de `9,5 mm`, concreto de `40 MPa`, aço estrutural de `345 MPa` e 10 barras longitudinais de `20 mm`.

In [2]:
concreto = ConcretoNormal(fck=40, modulo_elasticidade=30000)
aco_estrutural = AcoEstrutural(fy=345)
aco_armadura = AcoArmadura(fy=500)

pilar = PilarCircularPreenchido(
    diametro_tubo=600,
    espessura_tubo=9.5,
    material_aco_estrutural=aco_estrutural,
    material_concreto=concreto,
    material_armadura=aco_armadura,
    diametro_armadura_longitudinal=20,
    numero_armadura_longitudinal=10,
    diametro_armadura_transversal=5,
    espacamento_armadura_transversal=150,
    cobrimento=25,
    comprimento_pilar_destravado=3000,
)

print("Resumo resistente de cálculo")
print(f"NRd do pilar: {pilar.capacidade_axial_resistente_pilar_design / 1_000:,.2f} kN")
print(f"Mpl,Rd,xx: {pilar.momento_resistente_plastico_total_design_xx / 1_000_000:,.2f} kN.m")
print(f"Mpl,Rd,yy: {pilar.momento_resistente_plastico_total_design_yy / 1_000_000:,.2f} kN.m")

Resumo resistente de cálculo
NRd do pilar: 12,936.03 kN
Mpl,Rd,xx: 1,491.12 kN.m
Mpl,Rd,yy: 1,490.84 kN.m


## 3. Definição dos carregamentos

Cada caso é definido como uma tupla `(Nsd, Msd_x, Msd_y)`. A lista abaixo está em `kN` e `kN.m`; em seguida, convertemos para `N` e `N.mm`, que são as unidades usadas internamente.

In [3]:
carregamentos_kN_kNm = [
    (2_000, 100, 50),
    (6_000, 300, 250),
    (10_000, 500, 500),
    (13_000, 700, 600),
]

carregamentos = [
    (forca_kN * 1_000, momento_x_kNm * 1_000_000, momento_y_kNm * 1_000_000)
    for forca_kN, momento_x_kNm, momento_y_kNm in carregamentos_kN_kNm
]

for indice, (forca_kN, momento_x_kNm, momento_y_kNm) in enumerate(carregamentos_kN_kNm, start=1):
    print(
        f"Caso {indice}: "
        f"Nsd = {forca_kN:,.0f} kN | "
        f"Msd,x = {momento_x_kNm:,.0f} kN.m | "
        f"Msd,y = {momento_y_kNm:,.0f} kN.m"
    )

Caso 1: Nsd = 2,000 kN | Msd,x = 100 kN.m | Msd,y = 50 kN.m
Caso 2: Nsd = 6,000 kN | Msd,x = 300 kN.m | Msd,y = 250 kN.m
Caso 3: Nsd = 10,000 kN | Msd,x = 500 kN.m | Msd,y = 500 kN.m
Caso 4: Nsd = 13,000 kN | Msd,x = 700 kN.m | Msd,y = 600 kN.m


## 4. Verificação pelo Método I

O método retorna uma lista de booleanos: `True` indica que o caso passa e `False` indica que não passa. A função auxiliar abaixo apenas reproduz o índice de utilização para deixar a saída mais explicativa.

In [4]:
def calcular_indice_metodo_i(pilar, carregamentos, design=True):
    metodo = MetodoI()
    momento_adicional_x = metodo._calcular_momento_adicional_x(pilar, carregamentos)
    momento_adicional_y = metodo._calcular_momento_adicional_y(pilar, carregamentos)
    indices = []

    for indice, (forca_solicitante, momento_x, momento_y) in enumerate(carregamentos):
        if design:
            capacidade_axial = pilar.capacidade_axial_resistente_pilar_design
            momento_resistente_x = pilar.momento_resistente_plastico_total_design_xx
            momento_resistente_y = pilar.momento_resistente_plastico_total_design_yy
        else:
            capacidade_axial = pilar.capacidade_axial_resistente_pilar_nominal
            momento_resistente_x = pilar.momento_resistente_plastico_total_xx
            momento_resistente_y = pilar.momento_resistente_plastico_total_yy

        razao_axial = forca_solicitante / capacidade_axial
        momento_total_x = momento_x + momento_adicional_x[indice]
        momento_total_y = momento_y + momento_adicional_y[indice]

        if razao_axial >= 0.2:
            indice_utilizacao = razao_axial + (8 / 9) * (
                (momento_total_x / momento_resistente_x)
                + (momento_total_y / momento_resistente_y)
            )
        else:
            indice_utilizacao = (razao_axial / 2) + (
                (momento_total_x / momento_resistente_x)
                + (momento_total_y / momento_resistente_y)
            )

        indices.append(indice_utilizacao)

    return indices


metodo_i = MetodoI()
resultados = metodo_i.comparar_solicitacao(pilar, carregamentos, design=True)
indices = calcular_indice_metodo_i(pilar, carregamentos, design=True)

print("Resultado da verificação pelo Método I")
print("Caso | Nsd (kN) | Msd,x (kN.m) | Msd,y (kN.m) | Índice | Status")
print("-" * 76)

for indice, (carregamento, passou, indice_utilizacao) in enumerate(
    zip(carregamentos_kN_kNm, resultados, indices),
    start=1,
):
    forca_kN, momento_x_kNm, momento_y_kNm = carregamento
    status = "PASSA" if passou else "NÃO PASSA"
    print(
        f"{indice:>4} | "
        f"{forca_kN:>8,.0f} | "
        f"{momento_x_kNm:>13,.0f} | "
        f"{momento_y_kNm:>13,.0f} | "
        f"{indice_utilizacao:>6.3f} | "
        f"{status}"
    )

Resultado da verificação pelo Método I
Caso | Nsd (kN) | Msd,x (kN.m) | Msd,y (kN.m) | Índice | Status
----------------------------------------------------------------------------
   1 |    2,000 |           100 |            50 |  0.178 | PASSA
   2 |    6,000 |           300 |           250 |  0.792 | PASSA
   3 |   10,000 |           500 |           500 |  1.369 | NÃO PASSA
   4 |   13,000 |           700 |           600 |  1.780 | NÃO PASSA
